# Bài tập 2 — Làm sạch lần 2 & tạo đặc trưng → `df_clean.csv`
**Phụ trách:** Nguyễn Bá Công, Hà Trọng Hữu Duy · **Đầu vào:** `data/interim/df_raw.csv` · **Đầu ra:** `data/processed/df_clean.csv`

| Phần | Nội dung |
|---|---|
| 0 | Nạp dữ liệu & chẩn đoán |
| 1 | Baseline trên dữ liệu thô (mốc so sánh) |
| 2 | Bỏ cột rò rỉ nhãn và cột một giá trị |
| 3 | Bỏ tin thiếu nhãn loại hình |
| 4 | Trích xuất đặc trưng từ văn bản |
| 5 | Chuẩn hóa `loai_hinh` về 8 nhóm |
| 6 | Khử trùng lặp |
| 7 | Lọc ngoại lai hai tầng |
| 8 | Điền khuyết có kiểm soát |
| 9 | Che số trong văn bản |
| 10 | Biến phái sinh |
| 11 | Loại tin có sai số dự đoán thử lớn (APE > 50%) |
| 12 | Bảng kiểm toán, xuất file |

Mỗi bước ghi số dòng bị loại vào `bang_loai` → bảng ở mục 12 khớp với mục 4.1 của Data Dictionary và Project Charter (7.436 dòng).

In [1]:
import sys
from pathlib import Path

# Thư mục gốc dự án = thư mục cha của notebooks/
GOC = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(GOC))

import re
import numpy as np
import pandas as pd
from src.utils import doc_config

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
cfg = doc_config()
THU_MUC_THO = GOC / cfg['duong_dan']['du_lieu_tho']
THU_MUC_TRUNG_GIAN = GOC / cfg['duong_dan']['du_lieu_trung_gian']
THU_MUC_SACH = GOC / cfg['duong_dan']['du_lieu_sach']
print('Thư mục gốc:', GOC)

Thư mục gốc: /home/claude/repo


## 0. Nạp dữ liệu & chẩn đoán ban đầu

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

raw = pd.read_csv(THU_MUC_TRUNG_GIAN / 'df_raw.csv')
print(raw.shape)
raw.head(3)

(11888, 10)


,tieu_de,gia_ban,dien_tich_m2,tinh_thanh,quan_huyen,mo_ta_dac_diem,loai_hinh,nguon_du_lieu,url,gia_moi_m2_trieu
0,"Bán căn hộ 2PN 2WC 74,8m2 full NT The Sun Aven...",7000.0,74.8,TP. Hồ Chí Minh,Quận 2,"Căn hộ chung cư tại The Sun Avenue, Mai Chí Th...",căn hộ,cafeland,https://nhadat.cafeland.vn/ban-can-ho-2pn-2wc-...,93.582888
1,Bán nhà riêng: Bán nhà 100m2- 6.98 tỷ tại Vo V...,6980.0,100.0,TP. Hồ Chí Minh,Quận 6,"Bán nhà riêng An Lạc, TP. Hồ Chí Minh HẠ GIÁ G...",Nhà rieng,cafeland,https://nhadat.cafeland.vn/ban-nha-rieng-ban-n...,69.800000
2,Bán nhà riêng: Bán nhà mặt tiền đường Nhánh An...,8900.0,90.0,TP. Hồ Chí Minh,Quận 6,"Bán nhà riêng An Lạc, TP. Hồ Chí Minh Bán nhà ...",Nhà rieng,cafeland,https://nhadat.cafeland.vn/ban-nha-rieng-ban-n...,98.888889


In [3]:
print('Thieu du lieu:'); print(raw.isna().sum())
print()
print(raw[['gia_ban','dien_tich_m2','gia_moi_m2_trieu']].describe(percentiles=[.01,.25,.5,.75,.99]).round(2))

Thieu du lieu:
tieu_de                0
gia_ban                0
dien_tich_m2           0
tinh_thanh             0
quan_huyen             0
mo_ta_dac_diem         0
loai_hinh           2809
nguon_du_lieu          0
url                    0
gia_moi_m2_trieu       0
dtype: int64

        gia_ban  dien_tich_m2  gia_moi_m2_trieu
count  11888.00      11888.00          11888.00
mean    9568.22         81.01            126.53
std     7655.06         45.20             88.45
min      109.00         10.00              1.18
1%      1000.00         20.00             11.98
25%     4700.00         51.00             74.14
50%     6990.00         70.00            107.69
75%    11900.00        100.00            153.75
99%    38000.00        240.00            460.66
max    44000.00        270.00           2500.00


### Phát hiện 1 — `gia_moi_m2_trieu` là leakage tuyệt đối

Cột này bằng đúng `gia_ban / dien_tich_m2`. Nếu để lại trong X cùng `dien_tich_m2`,
mô hình chỉ cần nhân hai cột là ra nhãn → R² ≈ 1.0 nhưng vô dụng ngoài thực tế.

In [4]:
kiem_tra = (raw.gia_ban / raw.dien_tich_m2 - raw.gia_moi_m2_trieu).abs()
print('Ty le khop tuyet doi:', (kiem_tra < 0.01).mean())

Ty le khop tuyet doi: 1.0


### Phát hiện 2 — outlier, trùng lặp, truncation, lệch nguồn

In [5]:
ppsm = raw.gia_ban / raw.dien_tich_m2
print('Don gia max         :', round(ppsm.max(),1), 'trieu/m2  <-- phi ly')
print('So dong > 500 tr/m2 :', (ppsm > 500).sum())
print('So dong < 20 tr/m2  :', (ppsm < 20).sum())
print('Trung lap (gia,dt,quan,loai):', raw.duplicated(['gia_ban','dien_tich_m2','quan_huyen','loai_hinh']).sum())
print()
print('Truncation - gia_ban max :', raw.gia_ban.max(), '| so dong > 40000:', (raw.gia_ban>40000).sum())
print('Truncation - dien tich max:', raw.dien_tich_m2.max(), '| so dong > 250:', (raw.dien_tich_m2>250).sum())
print()
print('Lech giua 2 nguon (don gia median):')
print(ppsm.groupby(raw.nguon_du_lieu).median().round(1))

Don gia max         : 2500.0 trieu/m2  <-- phi ly
So dong > 500 tr/m2 : 76
So dong < 20 tr/m2  : 201
Trung lap (gia,dt,quan,loai): 1213

Truncation - gia_ban max : 44000.0 | so dong > 40000: 60
Truncation - dien tich max: 270.0 | so dong > 250: 66

Lech giua 2 nguon (don gia median):
nguon_du_lieu
batdongsan    103.4
cafeland      122.2
dtype: float64


### Phát hiện 3 — `quan_huyen` quá thô, phương sai nội bộ rất lớn

In [6]:
bang = ppsm.groupby(raw.quan_huyen).agg(['count','median','std']).round(1).sort_values('median')
bang

,count,median,std
quan_huyen,,,
Huyện Hóc Môn,307,44.9,142.2
TP. Thủ Đức,741,71.0,58.1
Quận 12,1281,73.4,35.3
Huyện Nhà Bè,130,75.7,30.2
Quận 9,524,80.0,35.3
Quận Bình Tân,877,93.3,36.3
Quận Tân Phú,859,103.3,42.7
Quận 6,293,103.3,50.1
Quận 4,280,111.8,120.9


### Phát hiện 4 — thông tin quyết định giá đang kẹt trong cột text

In [7]:
txt = raw.tieu_de.fillna('') + ' || ' + raw.mo_ta_dac_diem.fillna('')
low = txt.str.lower()
for ten, pat in [('so phong ngu', r'\d+\s*(?:phòng ngủ|pn\b)'), ('so tang', r'\d+\s*(?:tầng|lầu)'),
                 ('so hong/so do', r'sổ hồng|sổ đỏ'), ('trong hem', r'hẻm'),
                 ('ten duong', r'đường\s+\w'), ('CO GIA TRONG TEXT (leak)', r'\d+[\.,]?\d*\s*tỷ')]:
    print(f'{ten:28s}: {low.str.contains(pat, regex=True).mean():.1%}')

so phong ngu                : 78.2%
so tang                     : 73.2%
so hong/so do               : 49.9%
trong hem                   : 44.3%
ten duong                   : 82.6%
CO GIA TRONG TEXT (leak)    : 94.0%


## 1. Baseline trên dữ liệu thô
Random Forest chỉ với diện tích, quận, loại hình, nguồn — làm **mốc** để BT4 so sánh.

In [8]:
def mape(y, p):    return np.mean(np.abs((p - y) / y))
def medape(y, p):  return np.median(np.abs((p - y) / y))
X0 = pd.get_dummies(raw[['dien_tich_m2','quan_huyen','loai_hinh','nguon_du_lieu']])
y0 = raw.gia_ban
Xtr0, Xte0, ytr0, yte0 = train_test_split(X0, y0, test_size=.2, random_state=42)
pred0 = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr0, ytr0).predict(Xte0)
print(f'BASELINE  MAPE = {mape(yte0, pred0):.1%}   MedAPE = {medape(yte0, pred0):.1%}')


BASELINE  MAPE = 52.5%   MedAPE = 22.8%


---
# PIPELINE CHUẨN HÓA
## 2. Bỏ cột rò rỉ nhãn & cột một giá trị

In [9]:
df = raw.copy()
bang_loai = {}          # số dòng bị loại ở từng bước
nhat_ky = {'n_ban_dau': len(df)}

df = df.drop(columns=['gia_moi_m2_trieu',   # rò rỉ nhãn: = gia_ban / dien_tich_m2
                      'tinh_thanh'])        # chỉ có một giá trị
print(df.shape)

(11888, 8)


## 3. Bỏ tin thiếu nhãn loại hình
Toàn bộ dòng trống `loai_hinh` đến từ batdongsan. Gán chung vào "Khác" sẽ tạo một nhóm lớn trộn đủ loại nhà, làm sai so sánh giá theo loại hình, nên nhóm **loại bỏ**.

In [10]:
print('Nguồn của các dòng thiếu:', df.loc[df.loai_hinh.isna(), 'nguon_du_lieu'].value_counts().to_dict())
truoc = len(df)
df = df.dropna(subset=['loai_hinh'])
bang_loai['1. Thiếu nhãn loại hình'] = truoc - len(df)
print('Còn lại:', len(df))

Nguồn của các dòng thiếu: {'batdongsan': 2809}
Còn lại: 9079


## 4. Trích xuất đặc trưng từ text

Phần tạo ra phần lớn mức cải thiện. Gộp `tieu_de` + `mo_ta_dac_diem` rồi bóc ra
biến số (phòng ngủ, tầng, mặt tiền, phường, đường) và biến nhị phân (hẻm, sổ hồng, ô tô...).

In [11]:
from src.features.tao_dac_trung import trich_dac_trung_van_ban

# Regex chi tiết xem trong src/features/tao_dac_trung.py
df = trich_dac_trung_van_ban(df)
nhat_ky['ty_le_trich_duoc'] = {c: round(df[c].notna().mean(), 3) for c in
    ['so_phong_ngu','so_tang','so_wc','mat_tien_m','rong_hem_m','phuong','ten_duong']}
pd.Series(nhat_ky['ty_le_trich_duoc']).sort_values(ascending=False)

so_phong_ngu    0.732
so_tang         0.714
ten_duong       0.627
so_wc           0.383
phuong          0.333
mat_tien_m      0.209
rong_hem_m      0.114
dtype: float64

## 5. Chuẩn hóa `loai_hinh`
- `Nhà rieng` → `Nhà riêng`; `căn hộ` → `Căn hộ`
- `Kho - Nhà xưởng`, `Nhà hàng - Khách sạn`, `Bán nhà hàng - Khách sạn` → `Khác` (mỗi nhóm chỉ vài tin)

In [12]:
from src.features.tao_dac_trung import ANH_XA_LOAI_HINH

print('Trước:', df.loai_hinh.value_counts().to_dict())
df['loai_hinh'] = df.loai_hinh.str.lower().str.strip().map(ANH_XA_LOAI_HINH)
assert df.loai_hinh.notna().all(), 'Có nhãn chưa được ánh xạ'
df.loai_hinh.value_counts()

Trước: {'Nhà rieng': 4273, 'căn hộ': 1222, 'Nhà phố': 1181, 'Nhà mặt tiền': 989, 'Đất': 859, 'Biệt thự': 494, 'Shophouse': 48, 'Kho - Nhà xưởng': 8, 'Bán nhà hàng - Khách sạn': 3, 'Nhà hàng - Khách sạn': 2}


loai_hinh
Nhà riêng       4273
Căn hộ          1222
Nhà phố         1181
Nhà mặt tiền     989
Đất              859
Biệt thự         494
Shophouse         48
Khác              13
Name: count, dtype: int64

## 6. Khử trùng lặp
URL không trùng, nhưng cùng một căn có thể được nhiều môi giới đăng lại. Nếu giữ, bản sao rơi vào cả train và test → điểm test đẹp giả tạo.
**Hạn chế:** khóa 4 cột có thể gộp nhầm hai căn khác nhau nhưng trùng cả giá, diện tích, quận và loại hình.

In [13]:
truoc = len(df)
df = df.drop_duplicates(subset=['gia_ban','dien_tich_m2','quan_huyen','loai_hinh'], keep='first')
bang_loai['2. Trùng (giá, diện tích, quận, loại hình)'] = truoc - len(df)
print('Đã xóa', truoc - len(df), 'dòng trùng →', len(df))

Đã xóa 989 dòng trùng → 8090


## 7. Lọc outlier hai tầng

**Không cắt toàn cục** — Quận 1 và Hóc Môn chênh nhau hàng chục lần, một ngưỡng chung
sẽ xoá nhầm nhà bình thường ở quận đắt.

1. **Chặn cứng theo thị trường:** 15 – 450 triệu/m²
2. **Chặn mềm IQR trong từng quận:** ngoài $[Q_1 - 1.5\,\text{IQR},\ Q_3 + 1.5\,\text{IQR}]$
3. Diện tích ngoài 15 – 500 m²

In [14]:
from src.features.tao_dac_trung import loc_ngoai_lai_don_gia

truoc = len(df)
df = df[loc_ngoai_lai_don_gia(df)].copy()
bang_loai['3. Ngoại lai đơn giá (15–450 tr/m² và IQR theo quận)'] = truoc - len(df)

truoc = len(df)
df = df[df.dien_tich_m2.between(15, 500)]
bang_loai['4. Diện tích ngoài 15–500 m²'] = truoc - len(df)
print(bang_loai)
print('Còn lại:', len(df))

{'1. Thiếu nhãn loại hình': 2809, '2. Trùng (giá, diện tích, quận, loại hình)': 989, '3. Ngoại lai đơn giá (15–450 tr/m² và IQR theo quận)': 420, '4. Diện tích ngoài 15–500 m²': 7}
Còn lại: 7663


## 8. Điền khuyết có kiểm soát

Mỗi kiểu khuyết xử lý theo bản chất của nó, **không điền median bừa**:

| Cột | Cách điền | Lý do |
|---|---|---|
| `so_phong_ngu` | median theo `loai_hinh` | căn hộ và biệt thự phân bố rất khác nhau |
| `so_tang` | `1` | không nhắc tầng ≈ nhà trệt / đất |
| `so_wc` | suy từ số phòng ngủ | tương quan chặt |
| `mat_tien_m`, `rong_hem_m` | **sentinel −1** | "không ghi mặt tiền" tự nó là tín hiệu — điền median sẽ xoá mất |
| `phuong`, `ten_duong` | `Khong_ro` như một hạng mục | |

Thêm 2 cờ **missingness indicator** để mô hình phân biệt giá trị thật vs giá trị điền.

In [15]:
from src.features.tao_dac_trung import dien_khuyet

df = dien_khuyet(df)   # tạo co_tt_phong_ngu, co_tt_mat_tien trước khi điền
print('Còn thiếu:', df.isna().sum().sum())

Còn thiếu: 0


## 9. Chặn leakage trong text

**Phần lớn tin (mục 0, Phát hiện 4) chứa giá bán viết thẳng trong nội dung** ("Giá 9.8 tỷ", "3 tỷ 300 triệu").
Nếu TF-IDF / embedding cột gốc, mô hình đọc trộm đáp án.

Thay mọi chuỗi số kèm đơn vị bằng token `<SO>`, bỏ hai cột gốc.

In [16]:
mau_so = re.compile(r'\d+[\.,]?\d*\s*(tỷ|ty|triệu|trieu|tr/m2|tr\b|m²|m2)', re.I)
df['mo_ta_sach']  = df.mo_ta_dac_diem.fillna('').str.replace(mau_so, ' <SO> ', regex=True)
df['tieu_de_sach'] = df.tieu_de.fillna('').str.replace(mau_so, ' <SO> ', regex=True)

print('TRUOC:', df.mo_ta_dac_diem.iloc[0][:180])
print()
print('SAU  :', df.mo_ta_sach.iloc[0][:180])

TRUOC: Căn hộ chung cư tại The Sun Avenue, Mai Chí Thọ, An Phú, Quận 2, Hồ Chí Minh đang chờ chủ mới. Với diện tích 74,8m2 gồm 2PN + 2WC phù hợp cho những ai yêu thích không gian rộng rãi

SAU  : Căn hộ chung cư tại The Sun Avenue, Mai Chí Thọ, An Phú, Quận 2, Hồ Chí Minh đang chờ chủ mới. Với diện tích  <SO>  gồm 2PN + 2WC phù hợp cho những ai yêu thích không gian rộng rãi


## 10. Biến phái sinh

- `log_gia_ban` — target lệch phải nặng; train trên thang log rồi `exp()` ngược giảm ~10 điểm MAPE
- `log_dien_tich` — quan hệ giá–diện tích là phi tuyến
- `mat_do_xay` = số tầng × diện tích ≈ diện tích sàn sử dụng

In [17]:
from src.features.tao_dac_trung import THU_TU_COT

df['log_gia_ban']   = np.log(df.gia_ban)
df['log_dien_tich'] = np.log(df.dien_tich_m2)
df['mat_do_xay']    = df.so_tang * df.dien_tich_m2
df = df[THU_TU_COT].reset_index(drop=True)
print(df.shape)

(7663, 27)


## 11. Loại tin có sai số dự đoán thử lớn (APE > 50%)
Ở lần chạy gốc, nhóm huấn luyện thử Random Forest trên log(giá) với 80% dữ liệu (`train_test_split(random_state=42)`), dự đoán 20% còn lại và **loại 227 tin** có sai số tuyệt đối phần trăm trên 50% — xem là tin ảo hoặc thông tin sai lệch.

- Danh sách 227 URL được lưu ở `data/interim/loai_ape_lan_chay_goc.csv` để tái lập đúng tập dữ liệu trong Project Charter.
- Ô code thứ hai tái tạo bước này bằng hàm `tai_tao_buoc_ape`; vì lần chạy gốc không lưu phiên bản thư viện, kết quả **gần khớp**, không khớp tuyệt đối.

> ⚠️ **Rủi ro (Data Dictionary, vấn đề 9):** cả 227 tin đều nằm trong phần test của phép chia `random_state=42`. Ở BT4, **không** dùng lại đúng phép chia này để báo cáo điểm, vì các tin khó đoán đã bị loại khỏi tập test → điểm sẽ đẹp hơn thực tế. Nhóm sẽ đánh giá bằng cross-validation và báo cáo thêm điểm trên 7.663 dòng trước bước lọc này.

In [18]:
from src.features.tao_dac_trung import FILE_DANH_SACH_APE

danh_sach_ape = pd.read_csv(THU_MUC_TRUNG_GIAN / FILE_DANH_SACH_APE)
print('Số URL trong danh sách:', len(danh_sach_ape))
print('Tất cả đều có trong tập hiện tại:', danh_sach_ape.url.isin(df.url).all())
display(danh_sach_ape.groupby(['nguon_du_lieu', 'loai_hinh']).size().unstack(fill_value=0))

df_truoc_ape = df.copy()          # 7.663 dòng – giữ lại để BT4 so sánh
truoc = len(df)
df = df[~df.url.isin(danh_sach_ape.url)].reset_index(drop=True)
bang_loai['5. Sai số dự đoán thử APE > 50%'] = truoc - len(df)
print('Còn lại:', len(df))

Số URL trong danh sách: 227
Tất cả đều có trong tập hiện tại: True


loai_hinh,Biệt thự,Căn hộ,Khác,Nhà mặt tiền,Nhà phố,Nhà riêng,Shophouse,Đất
nguon_du_lieu,,,,,,,,
batdongsan,7,21,0,16,17,38,1,28
cafeland,4,38,1,3,1,46,0,6


Còn lại: 7436


In [19]:
# Kiểm chứng: tái tạo bước lọc bằng mô hình
from src.features.tao_dac_trung import tai_tao_buoc_ape

co_tai_tao = tai_tao_buoc_ape(df_truoc_ape)
url_tai_tao = df_truoc_ape.url[co_tai_tao]
print(f'Mô hình tái tạo đánh dấu {co_tai_tao.sum()} tin; trùng với danh sách gốc: '
      f'{url_tai_tao.isin(danh_sach_ape.url).sum()}/{len(danh_sach_ape)}')

Mô hình tái tạo đánh dấu 224 tin; trùng với danh sách gốc: 222/227


## 12. Bảng kiểm toán & xuất file

In [20]:
# Bảng kiểm toán: mọi dòng bị loại đều có lý do
kiem_toan = pd.DataFrame({'so_dong_loai': pd.Series(bang_loai)})
kiem_toan['ty_le_%_tren_df_raw'] = (kiem_toan.so_dong_loai / len(raw) * 100).round(2)
kiem_toan.loc['Tổng loại'] = [kiem_toan.so_dong_loai.sum(), round(kiem_toan.so_dong_loai.sum() / len(raw) * 100, 2)]
kiem_toan.loc['Giữ lại (df_clean)'] = [len(df), round(len(df) / len(raw) * 100, 2)]
assert kiem_toan.loc['Tổng loại', 'so_dong_loai'] + len(df) == len(raw)
kiem_toan

,so_dong_loai,ty_le_%_tren_df_raw
1. Thiếu nhãn loại hình,2809.0,23.63
"2. Trùng (giá, diện tích, quận, loại hình)",989.0,8.32
3. Ngoại lai đơn giá (15–450 tr/m² và IQR theo quận),420.0,3.53
4. Diện tích ngoài 15–500 m²,7.0,0.06
5. Sai số dự đoán thử APE > 50%,227.0,1.91
Tổng loại,4452.0,37.45
Giữ lại (df_clean),7436.0,62.55


In [21]:
# Kiểm tra chéo: hàm đóng gói trong src/ cho đúng kết quả như notebook
from src.features.tao_dac_trung import tao_df_clean
df_ham, _ = tao_df_clean(raw, in_nhat_ky=False, danh_sach_ape=danh_sach_ape)
pd.testing.assert_frame_equal(df, df_ham)
print('Notebook và src/features/tao_dac_trung.py cho cùng kết quả ✔')

Notebook và src/features/tao_dac_trung.py cho cùng kết quả ✔


In [22]:
THU_MUC_SACH.mkdir(parents=True, exist_ok=True)
df.to_csv(THU_MUC_SACH / 'df_clean.csv', index=False)
print('Đã lưu', THU_MUC_SACH / 'df_clean.csv', df.shape)

Đã lưu /home/claude/repo/data/processed/df_clean.csv (7436, 27)


## 13. Kiểm tra nhanh tập sạch (số liệu dùng trong Data Dictionary)

In [23]:
print('Ô trống:', df.isna().sum().sum(), '| URL trùng:', df.url.duplicated().sum())
print('Nguồn:', df.nguon_du_lieu.value_counts().to_dict())
print('Độ lệch gia_ban:', round(df.gia_ban.skew(), 2), '→ log:', round(df.log_gia_ban.skew(), 2))
print('Độ lệch dien_tich_m2:', round(df.dien_tich_m2.skew(), 2), '→ log:', round(df.log_dien_tich.skew(), 2))
df.describe().T.round(2)

Ô trống: 0 | URL trùng: 0
Nguồn: {'batdongsan': 4400, 'cafeland': 3036}
Độ lệch gia_ban: 1.76 → log: 0.09
Độ lệch dien_tich_m2: 1.46 → log: 0.05


,count,mean,std,min,25%,50%,75%,max
gia_ban,7436.0,9970.76,7413.22,340.00,5000.00,7500.00,12500.00,44000.00
log_gia_ban,7436.0,8.98,0.68,5.83,8.52,8.92,9.43,10.69
dien_tich_m2,7436.0,84.75,46.60,15.00,53.32,72.00,101.00,270.00
log_dien_tich,7436.0,4.31,0.51,2.71,3.98,4.28,4.62,5.60
so_phong_ngu,7436.0,3.47,1.64,1.00,3.00,3.00,4.00,10.00
so_tang,7436.0,2.58,1.59,1.00,1.00,2.00,4.00,8.00
so_wc,7436.0,3.18,1.25,1.00,2.00,3.00,4.00,8.00
mat_tien_m,7436.0,0.71,4.11,-1.00,-1.00,-1.00,-1.00,30.00
rong_hem_m,7436.0,-0.18,2.36,-1.00,-1.00,-1.00,-1.00,15.00
mat_do_xay,7436.0,216.36,197.46,15.00,87.00,159.00,280.00,2080.00


In [24]:
tam = df.assign(don_gia=df.gia_ban / df.dien_tich_m2)   # chỉ để mô tả, không lưu
bang_loai_hinh = tam.groupby('loai_hinh').agg(
    so_tin=('gia_ban', 'size'), trung_vi_gia=('gia_ban', 'median'),
    trung_vi_dien_tich=('dien_tich_m2', 'median'), trung_vi_don_gia=('don_gia', 'median'),
).sort_values('so_tin', ascending=False)
bang_loai_hinh.insert(1, 'ty_le_%', (bang_loai_hinh.so_tin / len(df) * 100).round(2))
bang_loai_hinh.round(2)

,so_tin,ty_le_%,trung_vi_gia,trung_vi_dien_tich,trung_vi_don_gia
loai_hinh,,,,,
Nhà riêng,3580,48.14,7225.0,65.0,118.14
Nhà phố,970,13.04,6985.0,64.0,115.05
Căn hộ,949,12.76,6000.0,70.0,78.43
Nhà mặt tiền,801,10.77,10500.0,88.0,121.17
Đất,680,9.14,6200.0,100.0,68.07
Biệt thự,425,5.72,14000.0,126.0,115.38
Shophouse,24,0.32,7500.0,102.0,62.05
Khác,7,0.09,16800.0,160.0,85.71


In [25]:
tam.groupby('quan_huyen').agg(so_tin=('gia_ban', 'size'), trung_vi_don_gia=('don_gia', 'median'))\
   .sort_values('trung_vi_don_gia', ascending=False).round(1)

,so_tin,trung_vi_don_gia
quan_huyen,,
Quận 5,89,208.1
Quận 3,246,196.6
Quận 10,254,181.6
Quận 1,606,175.7
Quận 11,163,140.6
Quận Bình Thạnh,460,136.9
Quận Tân Bình,456,135.4
Quận 4,127,121.9
Quận 2,368,121.9


---
## Hạn chế còn tồn tại
1. **`phuong` chỉ trích được khoảng 1/3 số tin** — giới hạn của nguồn, không phải của regex.
2. **Dải dữ liệu bị chặn:** giá tối đa 44 tỷ, diện tích tối đa 270 m² (giới hạn từ bước lọc ở `df_raw`). Mô hình không áp dụng cho BĐS ngoài dải này.
3. **Không có tọa độ.** Muốn giảm sai số đáng kể cần geocode `ten_duong` + `phuong`.
4. **Không có ngày đăng** trong `df_raw`/`df_clean` → dữ liệu là một lát cắt, không phân tích xu hướng.
5. **Khử trùng theo 4 cột** có thể gộp nhầm vài căn khác nhau (mục 6).
6. **Bước lọc APE (mục 11)** có thể gây thiên lệch chọn mẫu; BT4 phải đánh giá thêm trên `df_truoc_ape` (7.663 dòng).

### Hướng cho Bài tập 3–4
- Báo cáo cả MAPE và MedAPE; thử LightGBM trên thang log.
- Target-encode `ten_duong` theo K-fold; cân nhắc mô hình riêng cho nhóm `Đất`.

### Ghi chú sử dụng công cụ AI
Theo mục IV của đề cương, nhóm ghi rõ phần có AI hỗ trợ:
- **Claude (Anthropic)** hỗ trợ: rà soát tính tái lập, gom các bước làm sạch thành hàm trong `src/`, đổi đường dẫn tuyệt đối sang tương đối, viết bảng kiểm toán số dòng bị loại, tái tạo bước lọc APE để kiểm chứng.
- Logic xử lý (ngưỡng lọc, cách điền khuyết, regex trích đặc trưng) do nhóm thiết kế và giải thích được; mọi con số trong notebook được sinh ra khi chạy lại, không nhập tay.